# Spitex Standortplanung – Clustering Exploration

Dieses Notebook zeigt Schritt für Schritt, wie wir die Rasterzellen mit **K-Means Clustering** gruppieren.

Da es kein bekanntes Label gibt ("optimaler Spitex-Standort"), nutzen wir **unsupervised Learning**.
Das Clustering hilft uns, typische Gebietstypen in Zürich zu erkennen.

**Voraussetzung:** `python scripts/build_features.py` wurde ausgeführt.

**Ablauf:**
1. Daten laden und inspizieren
2. Preprocessing (Impute + Skalierung)
3. Silhouette-Scores für k=3..6 vergleichen
4. Finales Clustering mit k=4
5. Cluster-Profile und Heatmap
6. PCA-Visualisierung

In [ ]:
import sys
sys.path.append("..")

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
from sklearn.decomposition import PCA

from src.clustering import (
    CLUSTER_COLORS,
    CLUSTERING_FEATURES,
    DEFAULT_K,
    add_cluster_labels,
    build_cluster_profiles,
    prepare_features,
    run_kmeans_grid,
)

# Chart-Output-Ordner
CHARTS_DIR = Path("charts")
CHARTS_DIR.mkdir(exist_ok=True)

## 1. Daten laden

In [ ]:
# grid_features.geojson laden (erzeugt von scripts/build_features.py)
gdf = gpd.read_file("../data/processed/grid_features.geojson")
print(f"Geladen: {len(gdf)} Rasterzellen, {len(gdf.columns)} Spalten")
gdf[CLUSTERING_FEATURES].describe().round(3)

## 2. Preprocessing

Vor dem Clustering:
1. **Fehlende Werte** mit dem Median auffüllen
2. **Standardisierung** (StandardScaler) – damit alle Features gleichwertig behandelt werden

In [ ]:
# Features vorbereiten (Median-Impute + StandardScaler)
X_scaled, used_features = prepare_features(gdf)

print(f"Feature-Matrix: {X_scaled.shape}")
print(f"Mittelwert nach Skalierung (sollte ~0 sein): {X_scaled.mean(axis=0).round(3)}")

## 3. Silhouette-Score Vergleich: k=3, 4, 5, 6

Der **Silhouette-Score** misst, wie gut die Cluster voneinander getrennt sind (1.0 = perfekt, 0.0 = überlappend).

In [ ]:
# K-Means Grid Search
scores_df = run_kmeans_grid(X_scaled, k_values=[3, 4, 5, 6])
print(scores_df.to_string(index=False))

# Silhouette-Scores als Balkendiagramm
fig, ax = plt.subplots(figsize=(7, 4))
scores_plot = scores_df.sort_values("k")
colors = ["#e74c3c" if k == DEFAULT_K else "#3498db" for k in scores_plot["k"]]
bars = ax.bar(scores_plot["k"].astype(str), scores_plot["silhouette_score"], color=colors)
ax.set_xlabel("Anzahl Cluster (k)")
ax.set_ylabel("Silhouette Score")
ax.set_title("Silhouette-Score für verschiedene k-Werte")
ax.set_ylim(0, scores_plot["silhouette_score"].max() * 1.2)
for bar, score in zip(bars, scores_plot["silhouette_score"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
            f"{score:.4f}", ha="center", fontsize=9)
ax.legend([plt.Rectangle((0,0),1,1, color="#e74c3c")], [f"k={DEFAULT_K} (gewählt)"])
plt.tight_layout()
plt.savefig(CHARTS_DIR / "01_silhouette_scores.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Finales Clustering mit k=4

In [ ]:
# Cluster-Labels dem GeoDataFrame hinzufügen
gdf_clustered = add_cluster_labels(gdf, X_scaled, k=DEFAULT_K)

# Anzahl Zellen pro Cluster
counts = gdf_clustered["cluster"].value_counts().sort_index()
print("Anzahl Zellen pro Cluster:")
for cluster_id, n in counts.items():
    print(f"  Cluster {cluster_id}: {n} Zellen")

## 5. Cluster-Profile

Welche Merkmale charakterisieren jeden Cluster? Die Heatmap zeigt Z-Scores –
rot = überdurchschnittlich, blau = unterdurchschnittlich.

In [ ]:
# Cluster-Profile berechnen
profiles = build_cluster_profiles(gdf_clustered, used_features)

# Tabelle ausgeben
display_profiles = profiles.copy()
display_profiles["cluster"] = display_profiles["cluster"].apply(lambda x: f"Cluster {int(x)}")
display_profiles = display_profiles.set_index("cluster")
display_profiles.round(3)

In [ ]:
# Heatmap der Cluster-Profile (Z-Scores)
profile_vals = profiles.set_index("cluster")[used_features]
profile_z = (profile_vals - profile_vals.mean()) / profile_vals.std()

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(
    profile_z,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    ax=ax,
    linewidths=0.5,
    yticklabels=[f"Cluster {int(i)}" for i in profile_z.index],
    annot_kws={"size": 8},
)
ax.set_title("Cluster-Profile (Rot = überdurchschnittlich, Blau = unterdurchschnittlich)", fontsize=12)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.savefig(CHARTS_DIR / "02_cluster_profile_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. PCA-Visualisierung

K-Means arbeitet in einem 9-dimensionalen Raum. Mit **PCA** projizieren wir die Daten auf 2 Dimensionen,
um die Cluster-Trennung sichtbar zu machen. Jeder Punkt ist eine Rasterzelle.

In [ ]:
# PCA auf 2 Komponenten
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_
print(f"Erklärte Varianz: PC1={explained[0]:.1%}, PC2={explained[1]:.1%}, Total={sum(explained):.1%}")

cluster_labels_arr = gdf_clustered["cluster"].values

fig, ax = plt.subplots(figsize=(9, 7))
for cluster_id in sorted(set(cluster_labels_arr)):
    mask = cluster_labels_arr == cluster_id
    color = CLUSTER_COLORS.get(cluster_id, "#aaaaaa")
    ax.scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        c=color,
        label=f"Cluster {cluster_id} (n={mask.sum()})",
        alpha=0.5,
        s=12,
    )
ax.set_xlabel(f"PC1 ({explained[0]:.1%} Varianz)")
ax.set_ylabel(f"PC2 ({explained[1]:.1%} Varianz)")
ax.set_title("K-Means Cluster im PCA-Raum")
ax.legend()
plt.tight_layout()
plt.savefig(CHARTS_DIR / "03_pca_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

## Zusammenfassung

- k=4 hat den besten Silhouette-Score unter den getesteten Werten.
- Die vier Cluster beschreiben unterschiedliche Gebietstypen in Zürich (Zentrum, Randgebiet, älteres Wohngebiet, Wachstumsgebiet).
- `share_60plus` und `share_80plus` sind stark korreliert – beide messen Pflegebedarf.

**Nächste Schritte:** `streamlit run app.py` – Cluster in der Karte inspizieren.